# Phase H-B — locked multi-seed CIFAR-100 study

Runs the preregistered five seeds with no test-time tuning. Progress is one short line per task. A completed method is saved immediately to Drive; rerunning the study cell resumes it. FLY seed 1993 runs first and stops the study automatically if it differs from the 93.89 reference by more than 0.5 percentage points.

In [ ]:
# === Edit paths only; do not add/change model hyperparameters ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/crt-soho'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
PHASE_G_EVIDENCE_ZIP = f'{DRIVE_ROOT}/schur_locked_heldout_results.zip'
DRIVE_FEATURE_CACHE = f'{DRIVE_ROOT}/tsoho_cifar100_cache'
FEATURE_CACHE_DIR = '/content/tsoho_cifar100_cache'
OUTPUT_DIR = f'{DRIVE_ROOT}/phaseh_multiseed_outputs'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'


In [ ]:
# Setup runtime and repository.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Verify Phase G evidence and restore the shared feature cache from Drive.
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
evidence = Path(PHASE_G_EVIDENCE_ZIP)
assert evidence.is_file(), f'Upload the exact Phase G ZIP to: {evidence}'
assert file_sha256(evidence) == '9ecaa259deb998f36abdd8052145b17a0ce84adeeb2168b29a83c039868cbc77', 'Wrong Phase G ZIP'
print('Phase G evidence: PASS')
required = ('metadata.json', 'train.pt', 'test.pt')
drive_cache = Path(DRIVE_FEATURE_CACHE)
local_cache = Path(FEATURE_CACHE_DIR)
if all((drive_cache / name).is_file() for name in required):
    shutil.rmtree(local_cache, ignore_errors=True)
    started = time.time()
    shutil.copytree(drive_cache, local_cache)
    print(f'Feature cache restored from Drive in {time.time()-started:.1f}s')
else:
    print('No complete Drive cache; the next cell will extract it once.')


In [ ]:
# Extract only if cache restore was unavailable. tqdm output shows live image-batch progress.
required = ('metadata.json', 'train.pt', 'test.pt')
local_cache = Path(FEATURE_CACHE_DIR)
if not all((local_cache / name).is_file() for name in required):
    shutil.rmtree(local_cache, ignore_errors=True)
    if CHECKPOINT_SOURCE == 'google_drive':
        CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
    else:
        from huggingface_hub import hf_hub_download
        CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
    import kagglehub
    downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
    candidates = [downloaded, *downloaded.rglob('cifar-100')]
    cifar = next(path for path in candidates if (path/'train').is_file() and (path/'test').is_file() and (path/'meta').is_file())
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', str(cifar), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', '/content/phaseh_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', '1993', '--num-classes', '100', '--num-tasks', '10', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('Extracting frozen ViT features (live progress below)...', flush=True)
    subprocess.run(command, check=True)
    shutil.rmtree(DRIVE_FEATURE_CACHE, ignore_errors=True)
    shutil.copytree(local_cache, DRIVE_FEATURE_CACHE)
    print('Feature cache saved to Drive:', DRIVE_FEATURE_CACHE)
else:
    print('Using restored feature cache; extraction skipped.')


In [ ]:
# Correctness gate. Do not continue unless return code is zero.
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_phaseh_multiseed.py', 'tests/test_cached_replay_baselines.py', 'tests/test_experiment_runner.py']
completed = subprocess.run(command)
assert completed.returncode == 0, 'Phase H-B tests failed; stop and send the traceback.'
print('Phase H-B correctness gate: PASS')


In [ ]:
# Locked study. Safe to rerun after interruption: completed seed/method units resume from Drive.
command = [sys.executable, '-u', 'tools/phaseh_multiseed.py', '--manifest', 'configs/phaseh_cifar100_multiseed.json', '--phase-g-evidence-zip', PHASE_G_EVIDENCE_ZIP, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda']
print('Starting/resuming locked Phase H-B study...', flush=True)
completed = subprocess.run(command)
assert completed.returncode == 0, 'Runner failed; send the traceback and do not edit hyperparameters.'
if Path(OUTPUT_DIR, 'STOPPED_FLY_DISCREPANCY.json').is_file():
    print('STOP: FLY reference gate failed. Do not run further experiments.')
else:
    assert Path(OUTPUT_DIR, 'phaseh_summary.json').is_file()
    print('Phase H-B study: COMPLETE')


In [ ]:
# Show compact result and download the evidence bundle.
import pandas as pd
summary_path = Path(OUTPUT_DIR, 'phaseh_summary.json')
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    table = pd.DataFrame(summary['method_summaries'])
    columns = ['method', 'average_incremental_accuracy_mean', 'average_incremental_accuracy_std', 'final_accuracy_mean', 'final_accuracy_std', 'forgetting_mean', 'persistent_state_bytes_mean', 'exemplar_free']
    display(table[columns].sort_values('average_incremental_accuracy_mean', ascending=False))
    display(pd.DataFrame(summary['paired_differences']))
else:
    stopped = json.loads(Path(OUTPUT_DIR, 'STOPPED_FLY_DISCREPANCY.json').read_text())
    display(pd.DataFrame([stopped['fly_reference_gate']]))
archive = '/content/phaseh_multiseed_results.zip'
subprocess.run(['zip', '-qr', archive, OUTPUT_DIR], check=True)
from google.colab import files
files.download(archive)
